# Lab dl_04 · Attention from scratch

**Goal.** Implement scaled dot-product attention, causal masking and multi-head self-attention, and verify each piece against
`torch.nn.functional.scaled_dot_product_attention`.

### Theory recap ([Vaswani et al. 2017](https://arxiv.org/abs/1706.03762))
$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.$$
Row $i$ of the weight matrix is a probability distribution over positions $j$; output $i$ is the weighted average of the values.
Dividing by $\sqrt{d_k}$ keeps the logits' variance $\approx 1$ when $q,k$ have unit-variance entries
($\mathrm{Var}(\mathbf q\cdot\mathbf k) = d_k$), so the softmax does not saturate.

**Causal mask:** in a language model position $i$ may only look at $j\le i$. Set $S_{ij}=-\infty$ for $j>i$ **before** the softmax
so those weights become exactly 0 and each row still sums to 1.

**Multi-head:** project $X$ to $Q,K,V\in\mathbb R^{T\times d}$, split the last axis into $h$ heads of size $d/h$, attend in each head
independently, concatenate and apply an output projection. Parameters: $4d^2+4d$, independent of $h$.

**How this works.** Fill in every cell marked `# TODO` (they currently `raise NotImplementedError`). Run the notebook top to bottom with the working directory set to `labs/`. The last cell runs hidden checks on your functions and records the result in the Learning Center. Do the paper homework for the unlocking skill first — the lab assumes you can do the math by hand.

*This lab is original course material, inspired by Andrej Karpathy's open-source teaching code ([micrograd](https://github.com/karpathy/micrograd), [nn-zero-to-hero](https://github.com/karpathy/nn-zero-to-hero), [nanoGPT](https://github.com/karpathy/nanoGPT), [build-nanogpt](https://github.com/karpathy/build-nanogpt), [minbpe](https://github.com/karpathy/minbpe)). Watching the corresponding lecture after finishing is highly recommended.*

In [ ]:
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. Scaled dot-product attention
`attention(q, k, v, mask=None)` works for any leading batch dims: `q` (…, Tq, d), `k` (…, Tk, d), `v` (…, Tk, dv). `mask` is a **bool** tensor broadcastable to (…, Tq, Tk) with `True` = may attend. Return `(output, weights)`.

In [ ]:
def attention(q, k, v, mask=None):
    # TODO
    raise NotImplementedError


def causal_mask(T, device=None):
    # TODO: (T, T) bool, lower-triangular (True on and below the diagonal)
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
q, k, v = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
out, w = attention(q, k, v, causal_mask(5))
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("max diff vs torch:", (out - ref).abs().max().item())
plt.imshow(w[0].detach()); plt.colorbar(); plt.title("causal attention weights"); plt.show()

## 2. Why $\sqrt{d_k}$? An experiment
For growing $d_k$, draw unit-Gaussian $q, k$ and look at the variance of $q\cdot k$ and at how peaked the softmax over 16 keys gets
with and without scaling.

In [ ]:
for dk in [4, 16, 64, 256, 1024]:
    q, k = torch.randn(10000, dk), torch.randn(10000, 16, dk)
    s = torch.einsum("nd,nkd->nk", q, k)
    peak_raw = F.softmax(s, -1).max(-1).values.mean().item()
    peak_scaled = F.softmax(s / math.sqrt(dk), -1).max(-1).values.mean().item()
    print(f"d_k={dk:5d}  Var(q.k)={s.var().item():8.1f}  mean max-weight raw={peak_raw:.3f}  scaled={peak_scaled:.3f}")

## 3. Multi-head self-attention
Conventions (the checks rely on them): attributes `self.qkv = nn.Linear(d_model, 3 * d_model)` and `self.proj = nn.Linear(d_model, d_model)`; `q, k, v = self.qkv(x).split(d_model, dim=-1)`; heads via `.view(B, T, n_heads, d_head).transpose(1, 2)`; use your `attention` and `causal_mask` when `causal=True`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, causal=True):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO: (B, T, d) -> (B, T, d)
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
mha = MultiHeadAttention(32, 4)
x = torch.randn(2, 10, 32)
print(mha(x).shape, "params:", sum(p.numel() for p in mha.parameters()), "= 4d^2+4d =", 4 * 32**2 + 4 * 32)

### Reflect (on paper)
1. Write the shapes of every intermediate tensor in `MultiHeadAttention.forward` for $B=2, T=10, d=32, h=4$.
2. What happens to the output at position 0 of a causal layer, whatever the other tokens are? Verify numerically.
3. The score matrix costs $O(T^2)$ memory per head. For $T = 8192$, 32 heads, fp16, how many GiB per layer? (This is what FlashAttention avoids materialising.)

## Grade
Runs the hidden checks (fast, small inputs, fixed seeds) and reports to the local app.

In [ ]:
from lc_client import grade
grade("dl_04_attention", globals())